In [4]:
import pandas as pd

df = pd.read_csv('/content/WA_FnUseC_TelcoCustomerChurn.csv')

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [5]:
df.shape

(7043, 21)

In [7]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [8]:
df['Churn'].value_counts()

,count
Churn,
No,5174
Yes,1869


In [9]:
df['Churn'].value_counts(normalize=True) * 100

,proportion
Churn,
No,73.463013
Yes,26.536987


In [10]:
df['Churn'].value_counts()

,count
Churn,
No,5174
Yes,1869


In [11]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

df['TotalCharges'].isnull().sum()

np.int64(11)

In [12]:
df[df['TotalCharges'].isnull()][['tenure', 'TotalCharges', 'Churn']]

,tenure,TotalCharges,Churn
488,0,NaN,No
753,0,NaN,No
936,0,NaN,No
1082,0,NaN,No
1340,0,NaN,No
3331,0,NaN,No
3826,0,NaN,No
4380,0,NaN,No
5218,0,NaN,No
6670,0,NaN,No


In [13]:
df = df.dropna(subset=['TotalCharges'])

In [14]:
df.shape

(7032, 21)

In [15]:
df.isnull().sum()

,0
customerID,0
gender,0
SeniorCitizen,0
Partner,0
Dependents,0
tenure,0
PhoneService,0
MultipleLines,0
InternetService,0
OnlineSecurity,0


In [16]:
df = df.drop('customerID', axis=1)

In [17]:
df.shape

(7032, 20)

In [18]:
df.dtypes

,0
gender,object
SeniorCitizen,int64
Partner,object
Dependents,object
tenure,int64
PhoneService,object
MultipleLines,object
InternetService,object
OnlineSecurity,object
OnlineBackup,object


In [19]:
X = df.drop('Churn', axis=1)
y = df['Churn']

In [20]:
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (7032, 19)
y shape: (7032,)


In [21]:
y = y.map({'No': 0, 'Yes': 1})

In [22]:
y.value_counts()

,count
Churn,
0,5163
1,1869


In [23]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [24]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (5625, 19)
X_test: (1407, 19)
y_train: (5625,)
y_test: (1407,)


In [25]:
categorical_features = X.select_dtypes(include=['object']).columns
numerical_features = X.select_dtypes(exclude=['object']).columns

print("Categorical features:")
print(list(categorical_features))

print("\nNumerical features:")
print(list(numerical_features))

Categorical features:
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']

Numerical features:
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']


In [26]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

print("Preprocessor created successfully!")

Preprocessor created successfully!


In [27]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("X_train_processed shape:", X_train_processed.shape)
print("X_test_processed shape:", X_test_processed.shape)

X_train_processed shape: (5625, 45)
X_test_processed shape: (1407, 45)


In [28]:
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

logistic_model.fit(X_train_processed, y_train)

print("Logistic Regression trained successfully!")


Logistic Regression trained successfully!


In [29]:
y_pred = logistic_model.predict(X_test_processed)

In [30]:
print(y_pred[:20])

[0 1 0 0 0 0 0 0 1 0 1 0 1 0 0 1 1 1 0 0]


In [31]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

y_prob = logistic_model.predict_proba(X_test_processed)[:, 1]
roc_auc = roc_auc_score(y_test, y_prob)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-Score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy : 0.8038
Precision: 0.6485
Recall   : 0.5722
F1-Score : 0.6080
ROC-AUC  : 0.8359

Confusion Matrix:
[[917 116]
 [160 214]]

Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1033
           1       0.65      0.57      0.61       374

    accuracy                           0.80      1407
   macro avg       0.75      0.73      0.74      1407
weighted avg       0.80      0.80      0.80      1407



In [32]:
from sklearn.tree import DecisionTreeClassifier

decision_tree = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

decision_tree.fit(X_train_processed, y_train)

print("Decision Tree trained successfully!")

Decision Tree trained successfully!


In [33]:
y_pred_dt = decision_tree.predict(X_test_processed)

y_prob_dt = decision_tree.predict_proba(X_test_processed)[:, 1]

In [34]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

print(f"Accuracy : {accuracy_score(y_test, y_pred_dt):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_dt):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred_dt):.4f}")
print(f"F1-Score : {f1_score(y_test, y_pred_dt):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob_dt):.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_dt))

Accuracy : 0.7896
Precision: 0.6021
Recall   : 0.6150
F1-Score : 0.6085
ROC-AUC  : 0.8296

Confusion Matrix:
[[881 152]
 [144 230]]


In [35]:
from sklearn.ensemble import RandomForestClassifier

random_forest = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

random_forest.fit(X_train_processed, y_train)

print("Random Forest trained successfully!")

Random Forest trained successfully!


In [36]:
y_pred_rf = random_forest.predict(X_test_processed)

y_prob_rf = random_forest.predict_proba(X_test_processed)[:, 1]

In [37]:
print(f"Accuracy : {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_rf):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred_rf):.4f}")
print(f"F1-Score : {f1_score(y_test, y_pred_rf):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob_rf):.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

Accuracy : 0.7910
Precision: 0.6307
Recall   : 0.5160
F1-Score : 0.5676
ROC-AUC  : 0.8302

Confusion Matrix:
[[920 113]
 [181 193]]


In [38]:
results = pd.DataFrame({
    'Model': [
        'Logistic Regression',
        'Decision Tree',
        'Random Forest'
    ],
    'Accuracy': [
        accuracy_score(y_test, y_pred),
        accuracy_score(y_test, y_pred_dt),
        accuracy_score(y_test, y_pred_rf)
    ],
    'Precision': [
        precision_score(y_test, y_pred),
        precision_score(y_test, y_pred_dt),
        precision_score(y_test, y_pred_rf)
    ],
    'Recall': [
        recall_score(y_test, y_pred),
        recall_score(y_test, y_pred_dt),
        recall_score(y_test, y_pred_rf)
    ],
    'F1-Score': [
        f1_score(y_test, y_pred),
        f1_score(y_test, y_pred_dt),
        f1_score(y_test, y_pred_rf)
    ],
    'ROC-AUC': [
        roc_auc_score(y_test, y_prob),
        roc_auc_score(y_test, y_prob_dt),
        roc_auc_score(y_test, y_prob_rf)
    ]
})

results

,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Logistic Regression,0.803838,0.648485,0.572193,0.607955,0.835929
1,Decision Tree,0.789623,0.602094,0.614973,0.608466,0.829625
2,Random Forest,0.791045,0.630719,0.516043,0.567647,0.830180


In [39]:
from sklearn.pipeline import Pipeline

final_model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

final_model.fit(X_train, y_train)

print("Final model pipeline trained successfully!")

Final model pipeline trained successfully!


In [40]:
final_pred = final_model.predict(X_test)
final_prob = final_model.predict_proba(X_test)[:, 1]

In [41]:
print(f"Accuracy : {accuracy_score(y_test, final_pred):.4f}")
print(f"Precision: {precision_score(y_test, final_pred):.4f}")
print(f"Recall   : {recall_score(y_test, final_pred):.4f}")
print(f"F1-Score : {f1_score(y_test, final_pred):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, final_prob):.4f}")

Accuracy : 0.8038
Precision: 0.6485
Recall   : 0.5722
F1-Score : 0.6080
ROC-AUC  : 0.8359


In [42]:
# Get the trained preprocessing and classifier
trained_preprocessor = final_model.named_steps['preprocessor']
trained_classifier = final_model.named_steps['classifier']

# Get feature names after preprocessing
feature_names = trained_preprocessor.get_feature_names_out()

# Get model coefficients
coefficients = trained_classifier.coef_[0]

# Create a DataFrame
feature_importance = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients
})

# Sort by coefficient
feature_importance = feature_importance.sort_values(
    'Coefficient',
    ascending=False
)

feature_importance.head(10)

,Feature,Coefficient
3,num__TotalCharges,0.644014
36,cat__Contract_Month-to-month,0.613846
16,cat__InternetService_Fiber optic,0.590184
32,cat__StreamingTV_Yes,0.191113
43,cat__PaymentMethod_Electronic check,0.180671
35,cat__StreamingMovies_Yes,0.177076
18,cat__OnlineSecurity_No,0.164459
27,cat__TechSupport_No,0.142949
14,cat__MultipleLines_Yes,0.088164
0,num__SeniorCitizen,0.071011


In [43]:
feature_importance.sort_values(
    'Coefficient',
    ascending=True
).head(10)

,Feature,Coefficient
1,num__tenure,-1.352313
38,cat__Contract_Two year,-0.779030
15,cat__InternetService_DSL,-0.616132
2,num__MonthlyCharges,-0.541006
39,cat__PaperlessBilling_No,-0.300387
12,cat__MultipleLines_No,-0.293470
22,cat__OnlineBackup_No internet service,-0.283724
28,cat__TechSupport_No internet service,-0.283724
19,cat__OnlineSecurity_No internet service,-0.283724
17,cat__InternetService_No,-0.283724


In [44]:
top_positive = (
    feature_importance
    .sort_values('Coefficient', ascending=False)
    .head(10)
)

top_negative = (
    feature_importance
    .sort_values('Coefficient', ascending=True)
    .head(10)
)

print("Top 10 Positive Features:")
display(top_positive)

print("\nTop 10 Negative Features:")
display(top_negative)

Top 10 Positive Features:


,Feature,Coefficient
3,num__TotalCharges,0.644014
36,cat__Contract_Month-to-month,0.613846
16,cat__InternetService_Fiber optic,0.590184
32,cat__StreamingTV_Yes,0.191113
43,cat__PaymentMethod_Electronic check,0.180671
35,cat__StreamingMovies_Yes,0.177076
18,cat__OnlineSecurity_No,0.164459
27,cat__TechSupport_No,0.142949
14,cat__MultipleLines_Yes,0.088164
0,num__SeniorCitizen,0.071011



Top 10 Negative Features:


,Feature,Coefficient
1,num__tenure,-1.352313
38,cat__Contract_Two year,-0.779030
15,cat__InternetService_DSL,-0.616132
2,num__MonthlyCharges,-0.541006
39,cat__PaperlessBilling_No,-0.300387
12,cat__MultipleLines_No,-0.293470
22,cat__OnlineBackup_No internet service,-0.283724
28,cat__TechSupport_No internet service,-0.283724
19,cat__OnlineSecurity_No internet service,-0.283724
17,cat__InternetService_No,-0.283724
